# Modélisation Paludisme (XGBoost)
## Prédiction du Risque Épidémique

**Objectif** : Développer, entraîner et évaluer le modèle XGBoost de prédiction
du risque paludisme pour les 22 régions de Madagascar.

**Pipeline** :
1. Construction du dataset d'entraînement (26 features)
2. Analyse des features et sélection
3. Optimisation hyperparamètres (Optuna)
4. Entraînement XGBoost Classifier + Regressor
5. Évaluation (AUC, F1, Calibration)
6. Explicabilité SHAP
7. Analyse des erreurs régionales
8. Validation seuils UNICEF

---

In [ ]:
# ── Imports ──────────────────────────────────────────────────────
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path().resolve().parents[1]))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from datetime import date, timedelta

plt.rcParams.update({
    'figure.figsize': (14, 6),
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'DejaVu Sans',
})

UNICEF_BLUE   = '#00AEEF'
UNICEF_DARK   = '#374EA2'
UNICEF_RED    = '#E2231A'
UNICEF_GREEN  = '#80BD41'
UNICEF_ORANGE = '#F26A21'

print('Environnement prêt')

## 1. Construction du Dataset d'Entraînement


In [ ]:
# ── Génération dataset synthétique (ou réel via FeatureEngineer) ─
from ml.training_scripts.train_malaria import _generer_donnees_synthetiques_malaria
from src.models.malaria_predictor import MALARIA_FEATURE_NAMES

print('Construction dataset...')
X_raw, y_raw, feature_names = _generer_donnees_synthetiques_malaria()

df = pd.DataFrame(X_raw, columns=feature_names)
df['target_risque'] = y_raw
df['target_binaire'] = (y_raw >= 0.25).astype(int)

print(f'Dataset : {X_raw.shape[0]} samples × {X_raw.shape[1]} features')
print(f'   Prévalence risque (≥0.25) : {df["target_binaire"].mean()*100:.1f}%')
print(f'   Score moyen : {y_raw.mean():.3f} ± {y_raw.std():.3f}')
print(f'\n26 Features :')
for i, f in enumerate(feature_names):
    print(f'  {i+1:2d}. {f}')

In [ ]:
# ── Distribution de la variable cible ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogramme score continu
ax1 = axes[0]
n, bins, patches = ax1.hist(y_raw, bins=40, edgecolor='white', color=UNICEF_BLUE, alpha=0.8)
for seuil, couleur, label in [
    (0.25, 'orange', 'Risque moyen'),
    (0.50, 'red', 'Risque élevé'),
    (0.75, 'darkred', 'Très élevé')
]:
    ax1.axvline(x=seuil, color=couleur, linestyle='--', linewidth=2, label=label)
ax1.set_title('Distribution du Score de Risque (continu)', fontweight='bold')
ax1.set_xlabel('Score de risque [0, 1]')
ax1.set_ylabel('Fréquence')
ax1.legend(fontsize=8)

# Répartition classes
ax2 = axes[1]
niveaux = pd.cut(y_raw, bins=[0, 0.25, 0.50, 0.75, 1.0],
                 labels=['Faible', 'Moyen', 'Élevé', 'Très élevé'])
niveaux_counts = niveaux.value_counts()
colors_levels = [UNICEF_GREEN, 'gold', UNICEF_ORANGE, UNICEF_RED]
bars = ax2.bar(niveaux_counts.index, niveaux_counts.values,
               color=colors_levels, edgecolor='white', alpha=0.9)
for bar, val in zip(bars, niveaux_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f'{val}\n({val/len(y_raw)*100:.1f}%)',
             ha='center', va='bottom', fontsize=9, fontweight='bold')
ax2.set_title('Répartition par Niveau de Risque', fontweight='bold')
ax2.set_ylabel('Nombre de samples')

# Q-Q Plot normalité
ax3 = axes[2]
from scipy import stats as scipy_stats
scipy_stats.probplot(y_raw, dist='norm', plot=ax3)
ax3.set_title('Q-Q Plot — Distribution Score', fontweight='bold')
ax3.get_lines()[0].set(color=UNICEF_BLUE, markersize=3, alpha=0.5)
ax3.get_lines()[1].set(color=UNICEF_RED, linewidth=2)

plt.suptitle('Analyse de la Variable Cible — Score de Risque Paludisme',
             fontsize=13, fontweight='bold', color=UNICEF_DARK)
plt.tight_layout()
plt.savefig('../../data/processed/malaria_target_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Prétraitement et Split Train/Test


In [ ]:
# ── Preprocessing ────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split 80/20 stratifié
y_bins = np.digitize(y_raw, bins=[0.25, 0.50, 0.75])
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.20, random_state=42, stratify=y_bins
)

# Scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train : {X_train.shape} | Test : {X_test.shape}')
print(f'Prévalence train : {(y_train >= 0.25).mean()*100:.1f}%')
print(f'Prévalence test  : {(y_test  >= 0.25).mean()*100:.1f}%')

# Analyse des valeurs manquantes
df_train = pd.DataFrame(X_train, columns=feature_names)
missing = df_train.isnull().sum()
print(f'\nValeurs manquantes : {missing.sum()} (attendu: 0 — toutes imputées)')

# Vérification variance
low_var = df_train.columns[df_train.var() < 0.01].tolist()
print(f'Features à variance faible (<0.01) : {low_var if low_var else "aucune"}')

## 3. Importance des Features — Analyse Préliminaire


In [ ]:
# ── Corrélation features vs cible ───────────────────────────────
corr_target = df_train.corrwith(pd.Series(y_train[:len(df_train)], name='target'))
corr_target = corr_target.sort_values()

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Corrélation Pearson avec la cible
ax1 = axes[0]
colors_corr = [UNICEF_RED if v > 0 else UNICEF_BLUE for v in corr_target.values]
bars = ax1.barh(corr_target.index, corr_target.values,
                color=colors_corr, alpha=0.8, edgecolor='white')
ax1.axvline(x=0, color='black', linewidth=0.8)
ax1.set_title('Corrélation Pearson — Features vs Score Risque',
               fontweight='bold')
ax1.set_xlabel('Coefficient de corrélation')
for bar, val in zip(bars, corr_target.values):
    ax1.text(val + (0.003 if val >= 0 else -0.003), bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=7)

# Importance Random Forest (rapide)
ax2 = axes[1]
from sklearn.ensemble import RandomForestClassifier
y_clf_train = (y_train >= 0.25).astype(int)
rf_quick = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_quick.fit(X_train_sc, y_clf_train)
importances = pd.Series(rf_quick.feature_importances_, index=feature_names).sort_values()

colors_imp = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(importances)))
ax2.barh(importances.index, importances.values,
          color=colors_imp, alpha=0.85, edgecolor='white')
ax2.set_title('Importance Random Forest (100 arbres)', fontweight='bold')
ax2.set_xlabel('Importance moyenne')
# Top 5 annotation
top5 = importances.nlargest(5)
for feat in top5.index:
    val = importances[feat]
    ax2.text(val + 0.001, list(importances.index).index(feat),
             f'★ {val:.3f}', va='center', fontsize=8,
             color='darkred', fontweight='bold')

plt.suptitle('Analyse de l\'Importance des Features — Modèle Paludisme',
             fontsize=13, fontweight='bold', color=UNICEF_DARK)
plt.tight_layout()
plt.savefig('../../data/processed/malaria_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 features :')
for i, (f, v) in enumerate(importances.nlargest(10).items(), 1):
    print(f'  {i:2d}. {f:30s}: {v:.4f}')

## 4. Entraînement XGBoost — Comparaison de Configurations


In [ ]:
# ── Comparaison configurations XGBoost ──────────────────────────
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score
from sklearn.model_selection import cross_val_score

y_clf_train = (y_train >= 0.25).astype(int)
y_clf_test  = (y_test  >= 0.25).astype(int)

configs = {
    'XGB Léger\n(100 arbres, depth=4)': {
        'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1,
        'subsample': 0.8, 'colsample_bytree': 0.8,
        'scale_pos_weight': 3, 'use_label_encoder': False,
        'eval_metric': 'auc', 'random_state': 42, 'n_jobs': -1,
    },
    'XGB Standard\n(500 arbres, depth=6)': {
        'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.05,
        'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 5,
        'gamma': 0.1, 'reg_alpha': 0.1,
        'scale_pos_weight': 3, 'use_label_encoder': False,
        'eval_metric': 'auc', 'random_state': 42, 'n_jobs': -1,
    },
    'XGB Régularisé\n(300 arbres, dropout)': {
        'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.05,
        'subsample': 0.7, 'colsample_bytree': 0.7, 'min_child_weight': 8,
        'gamma': 0.3, 'reg_alpha': 0.5, 'reg_lambda': 2.0,
        'scale_pos_weight': 3, 'use_label_encoder': False,
        'eval_metric': 'auc', 'random_state': 42, 'n_jobs': -1,
    },
}

resultats_configs = {}
for nom, params in configs.items():
    print(f'Entraînement : {nom.replace(chr(10), " ")}...', end=' ')
    clf = XGBClassifier(**params)
    clf.fit(X_train_sc, y_clf_train, verbose=False)
    y_pred_proba = clf.predict_proba(X_test_sc)[:, 1]
    y_pred_bin   = (y_pred_proba >= 0.5).astype(int)
    auc = roc_auc_score(y_clf_test, y_pred_proba)
    f1  = f1_score(y_clf_test, y_pred_bin, zero_division=0)
    ap  = average_precision_score(y_clf_test, y_pred_proba)
    resultats_configs[nom] = {'clf': clf, 'auc': auc, 'f1': f1, 'ap': ap,
                               'proba': y_pred_proba}
    print(f'AUC={auc:.4f}  F1={f1:.4f}')

# Meilleur modèle
meilleur_nom = max(resultats_configs, key=lambda k: resultats_configs[k]['auc'])
print(f'\nMeilleure configuration : {meilleur_nom.replace(chr(10), " ")}')
print(f'   AUC-ROC : {resultats_configs[meilleur_nom]["auc"]:.4f}')

In [ ]:
# ── Courbes ROC et Précision-Rappel ─────────────────────────────
from sklearn.metrics import (roc_curve, precision_recall_curve,
                              auc as sklearn_auc)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

colors_roc = [UNICEF_BLUE, UNICEF_RED, UNICEF_GREEN]

for (nom, res), color in zip(resultats_configs.items(), colors_roc):
    # ROC
    fpr, tpr, _ = roc_curve(y_clf_test, res['proba'])
    axes[0].plot(fpr, tpr, color=color, linewidth=2,
                 label=f'{nom.replace(chr(10), " ")} (AUC={res["auc"]:.3f})')
    # PR
    prec, rec, _ = precision_recall_curve(y_clf_test, res['proba'])
    axes[1].plot(rec, prec, color=color, linewidth=2,
                 label=f'{nom.replace(chr(10), " ")} (AP={res["ap"]:.3f})')

# ROC
axes[0].plot([0,1],[0,1], 'k--', alpha=0.4, label='Aléatoire (AUC=0.5)')
axes[0].fill_between([0,1],[0,1],[0,1], alpha=0.02, color='gray')
axes[0].set_title('Courbes ROC — Comparaison Configurations', fontweight='bold')
axes[0].set_xlabel('Taux Faux Positifs')
axes[0].set_ylabel('Taux Vrais Positifs (Rappel)')
axes[0].legend(fontsize=8, loc='lower right')
axes[0].axhline(y=0.7, color='gray', linestyle=':', alpha=0.5)
axes[0].text(0.02, 0.72, 'Seuil UNICEF recall=0.70', fontsize=8, color='gray')

# PR
baseline_pr = y_clf_test.mean()
axes[1].axhline(y=baseline_pr, color='gray', linestyle='--', alpha=0.5,
                label=f'Baseline (prévalence={baseline_pr:.2f})')
axes[1].set_title('Courbes Précision-Rappel', fontweight='bold')
axes[1].set_xlabel('Rappel (Sensibilité)')
axes[1].set_ylabel('Précision')
axes[1].legend(fontsize=8, loc='upper right')

plt.suptitle('Évaluation Comparative des Configurations XGBoost',
             fontsize=13, fontweight='bold', color=UNICEF_DARK)
plt.tight_layout()
plt.savefig('../../data/processed/malaria_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Entraînement Final avec MalariaPredictor


In [ ]:
# ── Entraînement modèle de production ───────────────────────────
from src.models.malaria_predictor import MalariaPredictor
from ml.training_scripts.train_malaria import (
    _train_test_split_malaria, _cross_validate_malaria
)

print('🏋️  Entraînement MalariaPredictor (modèle de production)...')

# Split
X_tr, X_te, y_tr, y_te = _train_test_split_malaria(X_raw, y_raw)

# Scaling
scaler_prod = StandardScaler()

# Modèle
model = MalariaPredictor()
model._build_model()

X_tr_sc = scaler_prod.fit_transform(X_tr)
X_te_sc  = scaler_prod.transform(X_te)

# Entraînement avec validation
n_val = int(len(X_tr_sc) * 0.15)
model.fit(
    X_tr_sc[:-n_val], y_tr[:-n_val],
    feature_names=list(feature_names),
    y_clf=(y_tr[:-n_val] >= 0.25).astype(int),
    y_reg=y_tr[:-n_val] * 100,
    X_val=X_tr_sc[-n_val:],
    y_val_clf=(y_tr[-n_val:] >= 0.25).astype(int),
)
model._scaler = scaler_prod

# Évaluation
metriques = model.evaluate(X_te_sc, y_te)
print(f'\nMétriques MalariaPredictor :')
for k, v in metriques.items():
    print(f'   {k:25s}: {v}')

In [ ]:
# ── Cross-validation 5-fold ──────────────────────────────────────
print('Cross-validation 5-fold...')
cv_scores = _cross_validate_malaria(model, X_raw, y_raw, feature_names, scaler_prod)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scores par fold
ax1 = axes[0]
bars = ax1.bar(range(1, len(cv_scores)+1), cv_scores,
               color=[UNICEF_GREEN if s >= 0.75 else UNICEF_ORANGE if s >= 0.65
                      else UNICEF_RED for s in cv_scores],
               edgecolor='white', alpha=0.9)
ax1.axhline(y=cv_scores.mean(), color='black', linestyle='--', linewidth=2,
            label=f'Moyenne : {cv_scores.mean():.4f}')
ax1.axhline(y=0.75, color=UNICEF_DARK, linestyle=':', linewidth=1.5,
            label='Seuil cible UNICEF (0.75)')
ax1.axhline(y=0.70, color='gray', linestyle=':', linewidth=1,
            label='Seuil minimum UNICEF (0.70)')
for bar, val in zip(bars, cv_scores):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax1.set_xlabel('Fold')
ax1.set_ylabel('AUC-ROC')
ax1.set_title('Cross-Validation 5-Fold — AUC-ROC', fontweight='bold')
ax1.set_ylim(0.5, 1.0)
ax1.legend(fontsize=8)

# Distribution scores CV + stats
ax2 = axes[1]
ax2.boxplot(cv_scores, patch_artist=True,
            boxprops=dict(facecolor=UNICEF_BLUE, alpha=0.6),
            medianprops=dict(color=UNICEF_RED, linewidth=2),
            whiskerprops=dict(linewidth=1.5),
            positions=[1])
ax2.scatter([1]*len(cv_scores), cv_scores, color=UNICEF_DARK,
            zorder=5, s=80, alpha=0.8, label='Score par fold')
ax2.axhline(y=0.70, color='gray', linestyle='--', alpha=0.7)
ax2.axhline(y=0.75, color=UNICEF_DARK, linestyle='--', alpha=0.7)
ax2.set_xlim(0.5, 1.5)
ax2.set_ylim(0.5, 1.0)
ax2.set_xticks([1])
ax2.set_xticklabels(['CV 5-Fold'])
ax2.set_ylabel('AUC-ROC')
ax2.set_title(
    f'Distribution AUC-ROC\nMoy={cv_scores.mean():.4f} ± {cv_scores.std():.4f}',
    fontweight='bold'
)

plt.suptitle('Cross-Validation MalariaPredictor', fontsize=12,
             fontweight='bold', color=UNICEF_DARK)
plt.tight_layout()
plt.savefig('../../data/processed/malaria_cv_scores.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nCV AUC-ROC : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'   Intervalle de confiance 95% : [{cv_scores.mean()-2*cv_scores.std():.4f}, '
      f'{cv_scores.mean()+2*cv_scores.std():.4f}]')

## 6. Explicabilité SHAP — Valeurs et Visualisations


In [ ]:
# ── SHAP Summary Plot ────────────────────────────────────────────
try:
    import shap

    explainer_shap = shap.TreeExplainer(model._clf)
    X_sample = X_te_sc[:200]  # Sous-échantillon pour performance
    shap_values = explainer_shap.shap_values(X_sample)
    if isinstance(shap_values, list):
        sv = shap_values[1]
    else:
        sv = shap_values

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    # Summary plot (beeswarm)
    plt.sca(axes[0])
    shap.summary_plot(sv, X_sample, feature_names=list(feature_names),
                      plot_type='dot', show=False, max_display=20,
                      color_bar_label='Valeur de la feature')
    axes[0].set_title('SHAP Summary Plot — Impact sur le Score de Risque',
                      fontweight='bold', color=UNICEF_DARK)

    # Bar plot importance SHAP
    plt.sca(axes[1])
    shap_importances = pd.Series(
        np.abs(sv).mean(axis=0), index=feature_names
    ).nlargest(15)
    colors_shap = plt.cm.RdBu_r(np.linspace(0.2, 0.8, len(shap_importances)))
    shap_importances.plot.barh(ax=axes[1], color=colors_shap[::-1], alpha=0.85)
    axes[1].set_title('Importance SHAP — Mean(|SHAP|) Top 15', fontweight='bold')
    axes[1].set_xlabel('Importance SHAP moyenne')

    plt.suptitle('Analyse d\'Explicabilité SHAP — MalariaPredictor\n'
                 'Standard UNICEF de transparence algorithmique',
                 fontsize=12, fontweight='bold', color=UNICEF_DARK)
    plt.tight_layout()
    plt.savefig('../../data/processed/malaria_shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\n🔍 Top 5 features SHAP (requises pour rapports UNICEF) :')
    for i, (feat, val) in enumerate(shap_importances.head(5).items(), 1):
        print(f'  {i}. {feat:30s}: SHAP={val:.4f}')

except ImportError:
    print('SHAP non disponible — pip install shap')
except Exception as e:
    print(f'SHAP échoué : {e}')

## 7. Calibration des Probabilités


In [ ]:
# ── Courbe de calibration ────────────────────────────────────────
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import brier_score_loss

y_pred_raw   = model._predict_raw(X_te_sc)
y_clf_test   = (y_te >= 0.25).astype(int)
brier_raw    = brier_score_loss(y_clf_test, y_pred_raw)

# Calibration isotonique
clf_cal = CalibratedClassifierCV(model._clf, method='isotonic', cv=3)
clf_cal.fit(X_tr_sc[:-n_val], (y_tr[:-n_val] >= 0.25).astype(int))
y_pred_cal   = clf_cal.predict_proba(X_te_sc)[:, 1]
brier_cal    = brier_score_loss(y_clf_test, y_pred_cal)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Courbe de calibration
ax1 = axes[0]
for proba, label, color in [
    (y_pred_raw, f'Non calibré (Brier={brier_raw:.4f})', UNICEF_BLUE),
    (y_pred_cal, f'Calibré isotonic (Brier={brier_cal:.4f})', UNICEF_RED),
]:
    frac_pos, mean_pred = calibration_curve(y_clf_test, proba, n_bins=10)
    ax1.plot(mean_pred, frac_pos, marker='o', linewidth=2, color=color, label=label)

ax1.plot([0,1],[0,1], 'k--', alpha=0.5, label='Calibration parfaite')
ax1.fill_between([0,1],[0,0.1],[0.1,0.2], alpha=0.05, color='gray')
ax1.set_xlabel('Probabilité prédite')
ax1.set_ylabel('Fraction de vrais positifs')
ax1.set_title('Courbe de Calibration\n(Reliability Diagram)', fontweight='bold')
ax1.legend(fontsize=9)

# Distribution des probabilités
ax2 = axes[1]
ax2.hist(y_pred_raw[y_clf_test==0], bins=30, alpha=0.6, color=UNICEF_BLUE,
          label='Négatifs (risque faible)', density=True)
ax2.hist(y_pred_raw[y_clf_test==1], bins=30, alpha=0.6, color=UNICEF_RED,
          label='Positifs (risque élevé)', density=True)
ax2.axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Seuil 0.5')
ax2.set_xlabel('Score de probabilité prédit')
ax2.set_ylabel('Densité')
ax2.set_title('Séparation des Distributions\nNégatifs vs Positifs', fontweight='bold')
ax2.legend(fontsize=9)

plt.suptitle('Calibration et Séparation — MalariaPredictor',
             fontsize=12, fontweight='bold', color=UNICEF_DARK)
plt.tight_layout()
plt.savefig('../../data/processed/malaria_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Brier Score (non calibré) : {brier_raw:.4f}')
print(f'Brier Score (calibré)     : {brier_cal:.4f}')
print(f'Amélioration calibration  : {(brier_raw - brier_cal)/brier_raw*100:.1f}%')

## 8. Validation des Seuils UNICEF et Analyse des Erreurs


In [ ]:
# ── Évaluation complète via evaluate.py ─────────────────────────
from ml.training_scripts.evaluate import (
    evaluate_malaria_model, generate_evaluation_report
)

print('Évaluation complète (standards UNICEF)...')
eval_results = evaluate_malaria_model(
    model=model,
    X_test=X_te_sc,
    y_test=y_te,
    feature_names=list(feature_names),
)

# Affichage validation UNICEF
valid = eval_results['validation_unicef']
print(f'\nConformité UNICEF : {valid["score_conformite"]}%')
print('\nDétail des seuils :')
for metrique, detail in valid['details'].items():
    status_icon = '✅' if detail['statut'].startswith('✅') else '❌'
    print(f'  {status_icon} {metrique:20s}: {detail["valeur"]:6.4f} (seuil: {detail["seuil"]})')

# Matrice de confusion
clf_m = eval_results['classification']
print(f'\nMatrice de confusion :')
print(f'  TP={clf_m["tp"]:4d}  FP={clf_m["fp"]:4d}')
print(f'  FN={clf_m["fn"]:4d}  TN={clf_m["tn"]:4d}')
print(f'\n  Sensibilité (Recall) : {clf_m["recall"]:.4f}  ← critique pour détection épidémies')
print(f'  Spécificité          : {clf_m["specificity"]:.4f}')
print(f'  VPP (Precision)      : {clf_m["precision"]:.4f}')

# Sauvegarde rapport
rapport_path = generate_evaluation_report(
    eval_results,
    Path('../../ml/experiments/malaria_evaluation_report.json')
)
print(f'\nRapport sauvegardé → {rapport_path}')

In [ ]:
# ── Analyse par décile ───────────────────────────────────────────
deciles = eval_results['decile_analysis']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df_deciles = pd.DataFrame(deciles)

# Prévalence réelle vs prédite par décile (lift curve)
ax1 = axes[0]
x = df_deciles['decile']
width = 0.35
ax1.bar(x - width/2, df_deciles['prevalence_reelle'], width,
        label='Prévalence réelle', color=UNICEF_RED, alpha=0.8)
ax1.bar(x + width/2, df_deciles['score_moyen_predit'], width,
        label='Score prédit moyen', color=UNICEF_BLUE, alpha=0.8)
ax1.axhline(y=df_deciles['prevalence_reelle'].mean(), color='gray',
            linestyle='--', alpha=0.5, label='Baseline (moyenne)')
ax1.set_xlabel('Décile de score prédit (1=faible, 10=élevé)')
ax1.set_ylabel('Prévalence / Score moyen')
ax1.set_title('Analyse par Décile — Alignment Prédit/Réel', fontweight='bold')
ax1.legend(fontsize=9)
ax1.set_xticks(x)

# Lift curve
ax2 = axes[1]
baseline = df_deciles['prevalence_reelle'].mean()
lift = df_deciles['prevalence_reelle'] / baseline if baseline > 0 else df_deciles['prevalence_reelle']
ax2.plot(df_deciles['decile'], lift, marker='o', color=UNICEF_DARK,
          linewidth=2.5, markersize=8, label='Lift par décile')
ax2.axhline(y=1, color='gray', linestyle='--', alpha=0.7, label='Lift=1 (aléatoire)')
ax2.fill_between(df_deciles['decile'], 1, lift,
                  where=(lift >= 1), alpha=0.15, color=UNICEF_GREEN,
                  label='Gain vs aléatoire')
ax2.set_xlabel('Décile de score prédit')
ax2.set_ylabel('Lift (prévalence / baseline)')
ax2.set_title('Courbe de Lift — Gain vs Approche Aléatoire', fontweight='bold')
ax2.legend(fontsize=9)

plt.suptitle('Analyse par Décile — Discrimination du Modèle Paludisme',
             fontsize=12, fontweight='bold', color=UNICEF_DARK)
plt.tight_layout()
plt.savefig('../../data/processed/malaria_decile_lift.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Test de Prédiction — Exemple Région MDG-ATS


In [ ]:
# ── Test predict() complet ───────────────────────────────────────
import asyncio
from src.preprocessing.feature_engineering import FeatureEngineer

REGION_TEST = 'MDG-ATS'  # Atsinanana — Côte Est, très endémique

async def tester_prediction():
    engineer = FeatureEngineer()
    features = await engineer.build_malaria_features(REGION_TEST)
    return features

print(f'Test de prédiction pour {REGION_TEST}...')
features = asyncio.run(tester_prediction())

# Prédiction
prediction = model.predict(features, horizon_days=14)

print(f'\nRésultat de prédiction :')
print(f'   Région     : {REGION_TEST}')
print(f'   Score risque: {prediction["score_risque"]:.4f}')
print(f'   Niveau      : {prediction["niveau_risque"]}')
print(f'   Cas prévus 7j: {prediction.get("cas_prevus_7j", "N/A")}')
print(f'   Cas prévus 14j: {prediction.get("cas_prevus_14j", "N/A")}')
print(f'   Fiabilité   : {prediction["fiabilite_modele"]:.4f}')

print(f'\n🔍 Top contributeurs SHAP :')
for i, contrib in enumerate(prediction.get('top_contributeurs', [])[:5], 1):
    direction = '↑' if contrib.get('direction') == 'hausse_risque' else '↓'
    print(f'   {i}. {direction} {contrib["nom"]:30s}: {contrib.get("contribution_pct", 0):.1f}%')

## 10. Récapitulatif et Prochaines Étapes


In [ ]:
# ── Récapitulatif ────────────────────────────────────────────────
print('=' * 70)
print('RÉCAPITULATIF — MODÈLE MALARIA XGBoost')
print('=' * 70)
print(f'  Modèle   : MalariaPredictor v{model.MODEL_VERSION}')
print(f'  Features : {len(feature_names)} (climatiques, épi, géo, temporelles)')
print(f'  Samples  : {len(X_raw)} (synthétiques / réels)')
print(f'\n  Métriques finales :')
clf_m = eval_results['classification']
print(f'    AUC-ROC     : {clf_m["auc_roc"]:.4f}  (seuil UNICEF ≥ 0.75 ✅ ou ❌)')
print(f'    F1-Score    : {clf_m["f1_score"]:.4f}  (seuil UNICEF ≥ 0.65)')
print(f'    Recall      : {clf_m["recall"]:.4f}  (seuil UNICEF ≥ 0.70 — critique)')
print(f'    Précision   : {clf_m["precision"]:.4f}')
print(f'    Brier Score : {eval_results["calibration"]["brier_score"]:.4f}')
print(f'\n  Conformité UNICEF : {eval_results["validation_unicef"]["score_conformite"]}%')
print(f'\n  Artefacts générés :')
print('    data/processed/malaria_*.png (7 graphiques)')
print('    ml/experiments/malaria_evaluation_report.json')
print(f'\n  Prochaines étapes :')
print('    1. Entraîner sur données réelles DHIS2 (2021-2024)')
print('    2. Optimiser hyperparamètres avec Optuna (50 trials)')
print('    3. Valider backtesting sur 6 mois glissants')
print('    4. Intégration pipeline Celery (retraining mensuel)')
print('    5. Déploiement API FastAPI → /api/v1/paludisme/risque/{region_id}')